In [26]:
import os
import shutil
import cv2
import pandas as pd
import numpy as np
from tqdm import tqdm

# ================= 配置 =================
# 原始数据路径 (我们从之前的 CSV 和 图片读取)
RAW_CSV_PATH = "/root/autodl-tmp/severstal/train.csv"
RAW_IMG_DIR = "/root/autodl-tmp/severstal/train_images"

# 新的输出路径
OUTPUT_PATH = "/root/autodl-tmp/exp1/SEVERSTAL_SLICED"

# 切片配置
SLICE_W = 400   # 切片宽
SLICE_H = 256   # 切片高 (原图高)
OVERLAP = 0     # 不重叠，直接切 1600/400 = 4 份
IMG_W = 1600
IMG_H = 256
# =======================================

def rle2bbox(rle):
    # RLE 解码 (和之前一样)
    try:
        runs = np.array([int(x) for x in rle.split()])
        starts, lengths = runs[0::2], runs[1::2]
        mask = np.zeros(IMG_H * IMG_W, dtype=np.uint8)
        for s, l in zip(starts, lengths):
            mask[s - 1 : s + l - 1] = 1
        mask = mask.reshape((IMG_H, IMG_W), order='F') 
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        bboxes = []
        for c in contours:
            x, y, w, h = cv2.boundingRect(c)
            bboxes.append([x, y, w, h]) # 绝对坐标 xywh
        return bboxes
    except:
        return []

def main():
    if os.path.exists(OUTPUT_PATH): shutil.rmtree(OUTPUT_PATH)
    os.makedirs(os.path.join(OUTPUT_PATH, 'images/train'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, 'labels/train'), exist_ok=True)

    print("⏳ 读取 CSV...")
    df = pd.read_csv(RAW_CSV_PATH)
    df['ClassId'] = df['ClassId'] - 1 
    groups = df.groupby('ImageId')
    
    all_imgs = [x for x in os.listdir(RAW_IMG_DIR) if x.endswith('.jpg')]
    
    # 只需要一部分数据验证效果即可，全跑太慢 (比如取前 8000 张)
    # 如果想跑全量，把切片去掉
    # all_imgs = all_imgs[:8000] 

    print(f"🚀 开始切片处理 {len(all_imgs)} 张图片 -> 预计生成 {len(all_imgs)*4} 张切片...")
    
    count_saved = 0
    
    for img_name in tqdm(all_imgs):
        img_path = os.path.join(RAW_IMG_DIR, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
        
        # 1. 获取该图的所有 bbox (绝对坐标)
        bboxes = []
        if img_name in groups.groups:
            img_df = groups.get_group(img_name)
            for _, row in img_df.iterrows():
                if pd.notna(row['EncodedPixels']):
                    bs = rle2bbox(row['EncodedPixels'])
                    for b in bs:
                        # [class, x, y, w, h]
                        bboxes.append([row['ClassId']] + b)
        
        # 2. 开始切片 (0, 400, 800, 1200)
        for i, x_start in enumerate(range(0, IMG_W, SLICE_W)):
            x_end = x_start + SLICE_W
            
            # 裁剪图片
            slice_img = img[:, x_start:x_end, :]
            slice_name = f"{os.path.splitext(img_name)[0]}_s{i}.jpg"
            
            # 处理 Bbox
            slice_bboxes = []
            has_defect = False
            
            for b in bboxes:
                cls, bx, by, bw, bh = b
                # 计算 bbox 中心点
                cx, cy = bx + bw/2, by + bh/2
                
                # 如果中心点在当前切片内
                if x_start <= cx < x_end:
                    # 转换坐标到切片系
                    new_x = bx - x_start
                    # 边界截断处理
                    new_x = max(0, new_x)
                    new_w = min(bw, SLICE_W - new_x)
                    
                    # 转 YOLO 归一化
                    nx = (new_x + new_w/2) / SLICE_W
                    ny = (by + bh/2) / SLICE_H
                    nw = new_w / SLICE_W
                    nh = bh / SLICE_H
                    
                    if nw > 0 and nh > 0:
                        slice_bboxes.append(f"{cls} {nx:.6f} {ny:.6f} {nw:.6f} {nh:.6f}")
                        has_defect = True
            
            # 保存逻辑：
            # 为了提高训练效率，我们丢弃一部分纯黑背景（负样本），
            # 只要包含缺陷的，或者一定比例的负样本
            # 这里简单起见：保留所有切片，或者你可以加 if has_defect: 来只保留有缺陷的
            
            save_img_path = os.path.join(OUTPUT_PATH, 'images/train', slice_name)
            save_txt_path = os.path.join(OUTPUT_PATH, 'labels/train', slice_name.replace('.jpg', '.txt'))
            
            cv2.imwrite(save_img_path, slice_img)
            with open(save_txt_path, 'w') as f:
                f.write('\n'.join(slice_bboxes))
            
            count_saved += 1

    # 生成 Config
    yaml_content = f"path: {OUTPUT_PATH}\ntrain: images/train\nval: images/train\nnc: 4\nnames: ['1', '2', '3', '4']"
    with open(os.path.join(OUTPUT_PATH, "severstal_sliced.yaml"), 'w') as f:
        f.write(yaml_content)

    print(f"✅ 切片完成！生成了 {count_saved} 张训练样本。")
    print(f"配置文件: {os.path.join(OUTPUT_PATH, 'severstal_sliced.yaml')}")

if __name__ == "__main__":
    main()

⏳ 读取 CSV...
🚀 开始切片处理 12568 张图片 -> 预计生成 50272 张切片...


100%|██████████| 12568/12568 [03:10<00:00, 65.81it/s]

✅ 切片完成！生成了 50272 张训练样本。
配置文件: /root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_sliced.yaml


In [29]:
import os
import random

label_dir = "/root/autodl-tmp/exp1/SEVERSTAL_SLICED/labels/train"
files = os.listdir(label_dir)

# 随机抽 5 个文件看看内容
print(f"📂 标签文件夹里共有 {len(files)} 个文件")
print("-" * 30)

samples = random.sample(files, 5)
for f in samples:
    path = os.path.join(label_dir, f)
    with open(path, 'r') as file:
        content = file.read().strip()
        status = "✅ 有内容" if content else "⚠️ 空文件 (可能是负样本)"
        print(f"📄 {f}: {status}")
        if content:
            print(f"   -> 内容预览: {content[:50]}...") # 只看前50个字符
print("-" * 30)

📂 标签文件夹里共有 50272 个文件
------------------------------
📄 b0f641041_s2.txt: ⚠️ 空文件 (可能是负样本)
📄 ea87a105c_s2.txt: ⚠️ 空文件 (可能是负样本)
📄 c81c30ca0_s2.txt: ✅ 有内容
   -> 内容预览: 2 0.500000 0.867188 1.000000 0.265625
2 0.318750 0...
📄 30799f11c_s3.txt: ⚠️ 空文件 (可能是负样本)
📄 674be7ddd_s1.txt: ⚠️ 空文件 (可能是负样本)
------------------------------


In [31]:
from ultralytics import YOLO

# 新的切片数据配置
SLICED_YAML = "/root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_sliced.yaml"

print("🚀 开始 Exp 2.1: Sliced Severstal Pre-training...")

model = YOLO('yolo11n.pt') 

model.train(
    data=SLICED_YAML,
    epochs=30,             # 数据多了4倍，Epoch 减小
    batch=64,              # 4090 显存大，拉满
    imgsz=640,             # 现在 400x256 放到 640x640 里，变形很小！
    workers=8,
    project='result_exp1',
    name='2.1_Severstal_Sliced',
    device='0',
    exist_ok=True,
    val=False              # 依然关闭验证加速
)

print("\n🏆 切片版预训练完成！")

🚀 开始 Exp 2.1: Sliced Severstal Pre-training...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_sliced.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=2.1_Severstal_Sliced, nbs=64, nms=False, opset=No

In [32]:
from ultralytics import YOLO

# ================= 配置 =================
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 🔥 关键：加载刚才跑出来的“高分”预训练权重
PRETRAINED_WEIGHTS = "/root/autodl-tmp/exp1/result_exp1/2.1_Severstal_Sliced/weights/best.pt"

print("🚀 开始 Exp 3.1: 切片版权重的终极迁移 (Sliced Transfer)...")

# ================= Stage 1: 冻结骨干 (让 Head 适应 NEU) =================
print("\n--- Stage 1: Frozen Backbone ---")
model = YOLO(PRETRAINED_WEIGHTS)

model.train(
    data=NEU_YAML,
    epochs=15,
    batch=16,
    imgsz=640,
    project='Thesis_Exp',
    name='3.1_Sliced_Transfer_Stage1',
    device='0',
    freeze=10,             # 🔒 锁住刚才辛苦练好的骨干
    lr0=0.01,
    exist_ok=True,
    val=True
)

# ================= Stage 2: 全局微调 (解冻) =================
print("\n--- Stage 2: Unfreeze & Fine-tune ---")
# 加载 Stage 1 刚热身好的权重
stage1_weights = "Thesis_Exp/3.1_Sliced_Transfer_Stage1/weights/best.pt"
model_ft = YOLO(stage1_weights)

model_ft.train(
    data=NEU_YAML,
    epochs=40,             # 给足时间微调
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='3.1_Sliced_Transfer_Final', # 最终结果看这里
    device='0',
    freeze=0,              # 🔓 解锁
    lr0=0.0005,            # 🐢 慢速微调
    lrf=0.01,
    exist_ok=True,
    val=True
)

print("\n🏆 终极迁移实验结束！请查看最终 mAP。")

🚀 开始 Exp 3.1: 切片版权重的终极迁移 (Sliced Transfer)...

--- Stage 1: Frozen Backbone ---
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/autodl-tmp/exp1/result_exp1/2.1_Severstal_Sliced/weights/best.pt, momentum=0.937, mosaic=

In [34]:
import os
import glob
import yaml

# ================= 配置 =================
# 基于刚才切片后的数据路径
SLICED_ROOT = "/root/autodl-tmp/exp1/SEVERSTAL_SLICED"
IMG_DIR = os.path.join(SLICED_ROOT, "images/train")
LABEL_DIR = os.path.join(SLICED_ROOT, "labels/train")

# ================= 过滤逻辑 =================
print("🚀 正在筛选只包含缺陷的样本...")

# 获取所有 txt 文件
txt_files = glob.glob(os.path.join(LABEL_DIR, "*.txt"))

defect_imgs = []
empty_count = 0

for txt_path in txt_files:
    # 检查文件是否为空
    if os.path.getsize(txt_path) > 0:
        with open(txt_path, 'r') as f:
            content = f.read().strip()
            if len(content) > 0:
                # 找到对应的图片路径
                img_name = os.path.basename(txt_path).replace(".txt", ".jpg")
                if os.path.exists(os.path.join(IMG_DIR, img_name)):
                    defect_imgs.append(os.path.join(IMG_DIR, img_name))
            else:
                empty_count += 1
    else:
        empty_count += 1

print(f"📊 统计结果:")
print(f"   - 总文件数: {len(txt_files)}")
print(f"   - 缺陷样本 (保留): {len(defect_imgs)}")
print(f"   - 纯背景样本 (剔除): {empty_count}")

# ================= 生成 txt 列表文件 =================
train_list_path = os.path.join(SLICED_ROOT, "train_defects_only.txt")
# 🔥 修复1: 写入列表文件时指定 utf-8
with open(train_list_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(defect_imgs))

# ================= 生成 YAML =================
# 🔥 修复2: 去掉中文注释，避免兼容性问题
yaml_content = f"""
path: {SLICED_ROOT}
train: {train_list_path}
val: {train_list_path}
nc: 4
names: ['1', '2', '3', '4']
"""

yaml_path = os.path.join(SLICED_ROOT, "severstal_defects_only.yaml")
# 🔥 修复3: 写入 yaml 时指定 utf-8
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print(f"✅ 纯缺陷配置已生成: {yaml_path}")

🚀 正在筛选只包含缺陷的样本...
📊 统计结果:
   - 总文件数: 50272
   - 缺陷样本 (保留): 11511
   - 纯背景样本 (剔除): 38761
✅ 纯缺陷配置已生成: /root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_defects_only.yaml


In [35]:
from ultralytics import YOLO

# 1. 纯缺陷预训练配置
DEFECT_YAML = "/root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_defects_only.yaml"
# 2. 目标域配置
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

print("🚀 Step 1: 开始纯缺陷预训练 (Pre-training on Defects Only)...")
model = YOLO('yolo11n.pt')
model.train(
    data=DEFECT_YAML,
    epochs=30,             # 10轮足够了，因为数据变少了
    batch=64,
    imgsz=640,
    project='Thesis_Exp',
    name='2.2_Pretrain_DefectsOnly',
    device='0',
    exist_ok=True,
    val=False
)

# 获取权重
pretrain_weights = "Thesis_Exp/2.2_Pretrain_DefectsOnly/weights/best.pt"
print(f"\n✅ 预训练完成，权重: {pretrain_weights}")

print("\n🚀 Step 2: 迁移到 NEU-DET (Transfer)...")
# 直接用最佳实践：冻结骨干 -> 微调
model_ft = YOLO(pretrain_weights)
model_ft.train(
    data=NEU_YAML,
    epochs=40,
    batch=16,
    imgsz=640,
    project='Thesis_Exp',
    name='3.2_Transfer_DefectsOnly',
    device='0',
    freeze=10,      # 先锁住，保护刚才学到的纯粹缺陷特征
    lr0=0.005,      # 适中的学习率
    exist_ok=True,
    val=True
)

print("\n🏆 纯缺陷样本迁移实验结束！")

🚀 Step 1: 开始纯缺陷预训练 (Pre-training on Defects Only)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/SEVERSTAL_SLICED/severstal_defects_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=2.2_Pretrain_DefectsOnly, nbs=64, nms

In [2]:
from ultralytics import YOLO

# 1. 数据集配置
DS_YAML = "/root/autodl-tmp/exp1/Defect_Spectrum_YOLO/data.yaml"
NEU_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# ================= Exp 2.3: Defect Spectrum 预训练 =================
print("🚀 [Exp 2.3] 开始 Defect Spectrum 预训练...")
model = YOLO('yolo11n.pt')

model.train(
    data=DS_YAML,
    epochs=100,            # 数据少，多跑几轮防止欠拟合
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='2.3_DS_Pretrain',
    device='0',
    exist_ok=True,
    #val=False              # 预训练不需要验证
)

# 获取权重
pretrain_weights = "result_exp1/2.3_DS_Pretrain/weights/best.pt"
print(f"\n✅ 预训练完成，权重: {pretrain_weights}")

# ================= Exp 3.3: 迁移到 NEU-DET =================
print("\n🚀 [Exp 3.3] 迁移学习 (Defect Spectrum -> NEU-DET)...")

# 依然采用稳健的策略：冻结骨干 -> 微调
model_ft = YOLO(pretrain_weights)

model_ft.train(
    data=NEU_YAML,
    epochs=50,
    batch=16,
    imgsz=640,
    project='result_exp1',
    name='3.3_DS_Transfer',
    device='0',
    freeze=10,             # 保护学到的“好特征”
    lr0=0.005,
    exist_ok=True,
    val=True
)

print("\n🏆 Defect Spectrum 迁移实验结束！期待 mAP > 0.79！")

🚀 [Exp 2.3] 开始 Defect Spectrum 预训练...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/Defect_Spectrum_YOLO/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=2.3_DS_Pretrain, nbs=64, nms=False, opset=None, optimize=False, o

In [1]:
import os
from ultralytics import YOLO
from ultralytics.models.yolo.detect import DetectionTrainer
import torch
import torch.nn as nn
import numpy as np

# ================= 1. SA-LDP 引擎 (标准版) =================
class SA_LDP_Engine:
    def __init__(self, model, epsilon=50.0, delta=1e-5):
        self.model = model
        self.epsilon = epsilon
        self.delta = delta
        self.layer_roles = self._identify_layers()

    def _identify_layers(self):
        roles = {}
        for name, _ in self.model.named_parameters():
            if any(f"model.{i}." in name for i in range(10)): roles[name] = "backbone"
            elif "Detect" in name or "head" in name: roles[name] = "head"
            else: roles[name] = "neck"
        return roles

    def step(self, current_epoch):
        # 全量数据下，Warmup 可以稍微短一点，或者保持 3-5
        if current_epoch < 3: return

        current_device = next(self.model.parameters()).device
        sensitivities = []
        param_groups = []
        names_list = []

        for name, p in self.model.named_parameters():
            if p.requires_grad and p.grad is not None:
                sensitivities.append(p.grad.norm(2).item())
                param_groups.append(p)
                names_list.append(name)
        
        if not sensitivities: return

        factors = []
        for n in names_list:
            role = self.layer_roles.get(n, "neck")
            if role == "backbone": factors.append(0.5) 
            elif role == "head":   factors.append(2.0) 
            else:                  factors.append(1.0)
            
        factors = torch.tensor(factors, device=current_device)
        weights = factors / factors.sum() * len(factors) 
        
        idx = 0
        for p in param_groups:
            layer_eps = self.epsilon * weights[idx]
            # 全量数据梯度较稳，裁剪阈值可以用标准的 10.0
            clip_val = 10.0 
            torch.nn.utils.clip_grad_norm_(p, clip_val)
            c = np.sqrt(2 * np.log(1.25 / self.delta))
            sigma = c * clip_val / (layer_eps + 1e-8)
            p.grad.add_(torch.randn_like(p.grad) * sigma)
            idx += 1

# ================= 2. 隐私训练器 =================
class PrivacyTrainer(DetectionTrainer):
    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg, weights, verbose)
        self.privacy_engine = SA_LDP_Engine(model, epsilon=50.0) 
        return model

    def optimizer_step(self):
        self.scaler.unscale_(self.optimizer)
        self.privacy_engine.step(self.epoch)
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 10.0)
        self.scaler.step(self.optimizer)
        self.scaler.update()
        self.optimizer.zero_grad()
        if self.ema: self.ema.update(self.model)

# ================= 3. 执行 Exp 7 (全量隐私对比) =================
# 全量数据配置
FULL_YAML = "/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml"

# 关键：使用 Exp 2.3 (Defect Spectrum 预训练权重) 作为起点
DS_PRETRAIN = "/root/autodl-tmp/exp1/result_exp1/2.3_DS_Pretrain/weights/best.pt"

print("🚀 开始 Exp 7: 全量数据 (80%) 下的隐私鲁棒性验证...")

# --- Exp 7.1: Baseline + Privacy (Full Data) ---
# 这就是之前的 Exp 4 (SA-LDP)，我们重新跑一遍确保环境一致，或者你可以直接查 Exp 4 的数据
print("\n⚔️ [Exp 7.1] Baseline (ImageNet) + Privacy (Full Data)...")
trainer_base = PrivacyTrainer(overrides={
    'model': 'yolo11n.pt',
    'data': FULL_YAML,
    'epochs': 50, # 全量数据跑久一点
    'batch': 16,
    'imgsz': 640,
    'project': 'result_exp1', 
    'name': '7.1_Full_Privacy_Baseline',
    'device': '0',
    'exist_ok': True
})
trainer_base.train()

# --- Exp 7.2: Transfer + Privacy (Full Data) ---
# 这是全新的实验！看 Transfer 能否在全量数据加噪后逆袭
print("\n🛡️ [Exp 7.2] Transfer (Defect Spectrum) + Privacy (Full Data)...")
if os.path.exists(DS_PRETRAIN):
    trainer_ds = PrivacyTrainer(overrides={
        'model': DS_PRETRAIN,
        'data': FULL_YAML,
        'epochs': 50,
        'batch': 16,
        'imgsz': 640,
        'project': 'result_exp1',
        'name': '7.2_Full_Privacy_Transfer',
        'device': '0',
        'exist_ok': True,
        'freeze': 10 # 依然建议冻结前10层，保护“抗噪特征”
    })
    trainer_ds.train()
else:
    print("❌ 找不到 Exp 2.3 的预训练权重，请先运行相关实验。")

🚀 开始 Exp 7: 全量数据 (80%) 下的隐私鲁棒性验证...

⚔️ [Exp 7.1] Baseline (ImageNet) + Privacy (Full Data)...
Ultralytics 8.3.248 🚀 Python-3.10.8 torch-2.1.2+cu118 CUDA:0 (NVIDIA GeForce RTX 4090, 24111MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/root/autodl-tmp/exp1/NEU_DET_YOLO/neu_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=7.1_Full_Priva